# 03 — Decompose the pipeline
Goal: understand stage contracts and identify measured bottlenecks. Run in the cloned repository after environment setup. The runner executes the experiment and saves aggregate results; do not execute twice into the same directory.

In [ ]:
from pragmatiq import api
import inspect
for f in [api.synthesize, api.tokenize, api.pretrain, api.embed, api.probe]:
    print(f.__name__, inspect.signature(f))

## 1. Generation and tokenization
Raw events/profile/labels become field-aware key/value tokens, shards and an LMDB index. The vocabulary is fitted before encoding. Inspect the actual manifests after running the runner.

In [ ]:
from pathlib import Path
import json
root = Path('runs/research_decomposed')
for p in [root/'raw/manifest.json', root/'tok/shard_manifest.json']:
    if p.exists():
        print(p, p.read_text()[:4000])

## 2. Pretraining
The nano encoder learns masked values. Keep a fixed token budget; a 30-step smoke test is not convergence. Inspect the saved run files.

In [ ]:
for p in sorted((root/'runs/decomposed').rglob('*')):
    if p.is_file(): print(p.name, p.stat().st_size)

## 3. Embeddings and downstream evaluation
Full-history vectors are for representation inspection. The probe computes separate embeddings truncated at eval_ts. This is necessary but not sufficient for leakage-free evaluation: pretraining and tokenizer fitting also need historical boundaries.

In [ ]:
import pyarrow.parquet as pq
p=root/'embeddings.parquet'
if p.exists():
    print(pq.read_schema(p))
    print('rows:', pq.ParquetFile(p).metadata.num_rows)

## 4. Bottleneck analysis
Compare elapsed stage times, not GPU utilization. The probe includes additional embedding extraction, so do not double-count it as a pure classifier timing.

In [ ]:
import pandas as pd
r=json.loads(Path('research/results/03_pipeline_metrics.json').read_text())
t=pd.Series(r['stage_seconds'],name='seconds').sort_values(ascending=False)
display(t.to_frame().assign(share_pct=100*t/t.sum()))
t.plot.barh(title='Wall-clock time by pipeline stage')

## Conclusions to record
Which stage dominates? What scales with events, tokens, customers or optimizer steps? How would Spark, memory mapping and asynchronous loading change the bottleneck? Distinguish measured observations from proposed improvements.